# Malware Removal Runbook (No-Triage Path)

This notebook is designed for **rapid removal and containment** when you already have high-confidence evidence of malicious execution and do not trust user-mode diagnostics in Win32 due to spoofing risk.

## Important scope
- Focus: containment, removal, persistence teardown, and verification
- Assumption: malicious processes are already observed and known
- Triage is intentionally skipped
- Execution context: healthy WSL2 instance orchestrating Windows commands

## Safety notes
- Most removal commands require **elevated PowerShell (Administrator)**.
- This notebook defaults to `DRY_RUN=True` to avoid accidental disruption.
- Switch to execution mode only after validating IOC lists and target paths.
- If persistence reappears after reboot, escalate to full IR or host rebuild.

In [ ]:
from __future__ import annotations

import datetime as dt
import json
import shlex
import subprocess
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable, List

RUNBOOK_NAME = "malware_removal_no_triage"
TIMESTAMP = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = Path.cwd() / "removal_artifacts"
OUTPUT_DIR.mkdir(exist_ok=True)
LOG_FILE = OUTPUT_DIR / f"{RUNBOOK_NAME}_{TIMESTAMP}.log"
PS_SCRIPT_FILE = OUTPUT_DIR / f"{RUNBOOK_NAME}_{TIMESTAMP}.ps1"

# Keep true for safety until you intentionally switch to live execution.
DRY_RUN = True
STOP_ON_ERROR = True

print("Output directory:", OUTPUT_DIR)
print("Log file:", LOG_FILE)
print("PowerShell script path:", PS_SCRIPT_FILE)
print("DRY_RUN:", DRY_RUN)

## Privilege model (what must be elevated)

Because you are skipping triage and going directly to removal, use this privilege model:

1. **WSL user context**
   - Good for orchestrating and logging
   - Good for file hashing and offline notes
   - Not sufficient for reliable malware removal
2. **PowerShell (non-admin)**
   - Insufficient for complete remediation
   - Can fail silently on protected resources
3. **PowerShell (Administrator)**
   - Required for Defender remediation, service/task control, registry cleanup, and forced process/service termination in many cases
   - Required for fast, deterministic cleanup when spoofing risk exists

For this notebook: assume commands marked `requires_admin=True` must be run in elevated PowerShell.

In [ ]:
@dataclass
class Step:
    name: str
    command: str
    requires_admin: bool = True
    critical: bool = True


def _powershell_command(ps: str) -> List[str]:
    return [
        "powershell.exe",
        "-NoProfile",
        "-ExecutionPolicy",
        "Bypass",
        "-Command",
        ps,
    ]


def run_powershell(ps: str, dry_run: bool = True) -> tuple[int, str, str]:
    cmd = _powershell_command(ps)
    if dry_run:
        return 0, "DRY_RUN: " + " \
"

    proc = subprocess.run(cmd, text=True, capture_output=True)
    return proc.returncode, proc.stdout, proc.stderr


def append_log(message: str) -> None:
    with LOG_FILE.open("a", encoding="utf-8") as f:
        f.write(message + "\n")


def execute_steps(steps: Iterable[Step], dry_run: bool = True, stop_on_error: bool = True) -> list[dict]:
    results = []
    for i, step in enumerate(steps, start=1):
        header = f"[{i:02d}] {step.name} | admin={step.requires_admin} | critical={step.critical}"
        print(header)
        append_log(header)

        rc, out, err = run_powershell(step.command, dry_run=dry_run)
        result = {
            "index": i,
            "name": step.name,
            "requires_admin": step.requires_admin,
            "critical": step.critical,
            "returncode": rc,
            "stdout": out,
            "stderr": err,
        }
        results.append(result)

        append_log("RETURN CODE: " + str(rc))
        if out.strip():
            append_log("STDOUT:\n" + out.strip())
        if err.strip():
            append_log("STDERR:\n" + err.strip())
        append_log("-" * 80)

        if rc != 0 and step.critical and stop_on_error:
            print("Stopping due to critical step failure:", step.name)
            break

    return results

## Configure known malicious indicators

Fill these lists with known malicious process names, service names, scheduled task names, registry values, and file paths observed in your environment.

This runbook is intentionally explicit and deterministic: **you provide known-bad indicators, then force teardown**.

In [ ]:
KNOWN_BAD_PROCESSES = [
    # Example placeholders
    # "badproc.exe",
]

KNOWN_BAD_SERVICES = [
    # "BadService",
]

KNOWN_BAD_TASKS = [
    # "\\Microsoft\\Windows\\Update\\BadTask",
]

KNOWN_BAD_PATHS = [
    # r"C:\\ProgramData\\BadFolder",
    # r"C:\\Users\\Public\\bad.exe",
]

KNOWN_BAD_RUN_VALUES = [
    # (r"HKCU:\\Software\\Microsoft\\Windows\\CurrentVersion\\Run", "BadValue"),
    # (r"HKLM:\\Software\\Microsoft\\Windows\\CurrentVersion\\Run", "BadValue"),
]

print("Configured process IOCs:", len(KNOWN_BAD_PROCESSES))
print("Configured service IOCs:", len(KNOWN_BAD_SERVICES))
print("Configured task IOCs:", len(KNOWN_BAD_TASKS))
print("Configured path IOCs:", len(KNOWN_BAD_PATHS))
print("Configured run-value IOCs:", len(KNOWN_BAD_RUN_VALUES))

## Build full removal sequence (containment -> kill -> persistence removal -> scan -> verify)

This section creates an ordered sequence of PowerShell steps.

Removal philosophy in this no-triage mode:
1. Contain communication and prevent further execution
2. Kill known malicious runtime objects
3. Disable/delete persistence mechanisms
4. Remove known malicious files
5. Run Defender remediation scans
6. Reboot and validate no recurrence

In [ ]:
def _quoted_ps_strings(items: list[str]) -> str:
    if not items:
        return "@()"
    escaped = [s.replace("'", "''") for s in items]
    return "@(" + ",".join(f"'{x}'" for x in escaped) + ")"


proc_array = _quoted_ps_strings(KNOWN_BAD_PROCESSES)
svc_array = _quoted_ps_strings(KNOWN_BAD_SERVICES)
task_array = _quoted_ps_strings(KNOWN_BAD_TASKS)
path_array = _quoted_ps_strings(KNOWN_BAD_PATHS)

run_entries = []
for key, value_name in KNOWN_BAD_RUN_VALUES:
    k = key.replace("'", "''")
    v = value_name.replace("'", "''")
    run_entries.append(f"@{{Key='{k}';Value='{v}'}}")
run_array = "@(" + ",".join(run_entries) + ")" if run_entries else "@()"

steps = [
    Step(
        name="Start transcript and harden script error behavior",
        command=(
            "$ErrorActionPreference='Stop'; "
            "$ProgressPreference='SilentlyContinue'; "
            "Start-Transcript -Path $env:TEMP\\malware_removal_transcript.txt -Force"
        ),
    ),
    Step(
        name="Optional emergency network containment (disable adapters except loopback)",
        command=(
            "Get-NetAdapter | Where-Object {$_.Status -eq 'Up' -and $_.HardwareInterface -eq $true} "
            "| ForEach-Object { Disable-NetAdapter -Name $_.Name -Confirm:$false }"
        ),
        critical=False,
    ),
    Step(
        name="Stop known malicious processes",
        command=(
            f"$names={proc_array}; "
            "foreach ($n in $names) { Get-Process -Name ([System.IO.Path]::GetFileNameWithoutExtension($n)) -ErrorAction SilentlyContinue | Stop-Process -Force -ErrorAction SilentlyContinue }"
        ),
    ),
    Step(
        name="Stop and disable known malicious services",
        command=(
            f"$svcs={svc_array}; "
            "foreach ($s in $svcs) { Stop-Service -Name $s -Force -ErrorAction SilentlyContinue; Set-Service -Name $s -StartupType Disabled -ErrorAction SilentlyContinue }"
        ),
    ),
    Step(
        name="Disable and unregister known malicious scheduled tasks",
        command=(
            f"$tasks={task_array}; "
            "foreach ($t in $tasks) { Disable-ScheduledTask -TaskPath ($t.Substring(0, $t.LastIndexOf('\\')+1)) -TaskName ($t.Substring($t.LastIndexOf('\\')+1)) -ErrorAction SilentlyContinue; Unregister-ScheduledTask -TaskPath ($t.Substring(0, $t.LastIndexOf('\\')+1)) -TaskName ($t.Substring($t.LastIndexOf('\\')+1)) -Confirm:$false -ErrorAction SilentlyContinue }"
        ),
    ),
    Step(
        name="Remove known malicious Run key values",
        command=(
            f"$items={run_array}; "
            "foreach ($i in $items) { Remove-ItemProperty -Path $i.Key -Name $i.Value -Force -ErrorAction SilentlyContinue }"
        ),
    ),
    Step(
        name="Delete known malicious files and directories",
        command=(
            f"$paths={path_array}; "
            "foreach ($p in $paths) { if (Test-Path $p) { Takeown /f $p /r /d y | Out-Null; Icacls $p /grant Administrators:F /t /c | Out-Null; Remove-Item -LiteralPath $p -Recurse -Force -ErrorAction SilentlyContinue } }"
        ),
    ),
    Step(
        name="Update Defender signatures",
        command="Update-MpSignature",
    ),
    Step(
        name="Run Defender quick scan for immediate active threat cleanup",
        command="Start-MpScan -ScanType QuickScan",
    ),
    Step(
        name="Run Defender full scan for broad filesystem coverage",
        command="Start-MpScan -ScanType FullScan",
    ),
    Step(
        name="Collect Defender detections after cleanup",
        command="Get-MpThreatDetection | Sort-Object InitialDetectionTime -Descending | Select-Object -First 50 | Format-Table -Auto",
        critical=False,
    ),
    Step(
        name="Stop transcript",
        command="Stop-Transcript",
        critical=False,
    ),
]

print(f"Prepared {len(steps)} removal steps.")

## Export a standalone PowerShell script

This cell writes a `.ps1` file with the same commands so you can run it directly in an elevated Windows PowerShell session.

Recommended usage:
1. Open PowerShell as Administrator
2. Review generated script
3. Execute with execution policy bypass for this run only

In [ ]:
script_lines = [
    "$ErrorActionPreference = 'Stop'",
    "$ProgressPreference = 'SilentlyContinue'",
    "",
]
for step in steps:
    script_lines.append(f"# {step.name}")
    script_lines.append(step.command)
    script_lines.append("")

PS_SCRIPT_FILE.write_text("\n".join(script_lines), encoding="utf-8")
print("Exported:", PS_SCRIPT_FILE)
print("Run from elevated PowerShell:")
print(f"powershell -ExecutionPolicy Bypass -File \"{PS_SCRIPT_FILE}\"")

## Execute from notebook (optional)

Behavior:
- `DRY_RUN=True`: prints commands that would run and writes structured logs
- `DRY_RUN=False`: executes PowerShell commands from WSL

If execution fails because elevation is required, run the exported script in an elevated PowerShell window.

In [ ]:
results = execute_steps(steps, dry_run=DRY_RUN, stop_on_error=STOP_ON_ERROR)

summary = {
    "total": len(results),
    "failed": sum(1 for r in results if r["returncode"] != 0),
    "dry_run": DRY_RUN,
    "timestamp": TIMESTAMP,
}

summary_file = OUTPUT_DIR / f"{RUNBOOK_NAME}_{TIMESTAMP}_summary.json"
summary_file.write_text(json.dumps(summary, indent=2), encoding="utf-8")

print(json.dumps(summary, indent=2))
print("Summary file:", summary_file)
print("Detailed log:", LOG_FILE)

## Post-removal hardening and closure checklist

After successful removal, complete these actions in order:

1. Reboot host and re-run quick validation scan
2. Confirm no known bad process/service/task reappears
3. Confirm no known bad files regenerate in deleted paths
4. Review Defender detections for successful remediation status
5. Rotate credentials used on infected host (local admin, browser-saved, tokens)
6. Patch OS and high-risk software immediately
7. Re-enable networking only after clean verification
8. Archive logs and script artifacts for incident record

If indicators recur after reboot or offline scan, escalate to full incident response and consider host rebuild.